# Satellite Analysis

## Download images using hex map coordinates

Docs:  
https://developers.google.com/maps/documentation/maps-static/start

In [ ]:
import json
import numpy as np
import requests

from os import makedirs, path
from PIL import Image as PImage
from time import sleep

from env import GOOGLE_API_KEY

def pad(n, l=3):
  return (l*"0" + str(n))[-l:] if n < 10**l else str(n)

In [ ]:
MAPS_URL = "https://maps.googleapis.com/maps/api/staticmap"
IMG_SIZE = 512

MAPS_PARAMS = {
  "size": f"{IMG_SIZE}x{IMG_SIZE}",
  "scale": 1,
  "format": "jpg",
  "maptype": "satellite",
  "key": GOOGLE_API_KEY,
}

MAPS_HEADER = {
  "User-Agent": "Mozilla/5.0"
}

run_name = f"sat_hex"

IMGS_DIR = "./satellites/imgs"
DATA_DIR = "./satellites/data"
SAT_IMGS_DIR = f"{IMGS_DIR}/{run_name}"

makedirs(DATA_DIR, exist_ok=True)
makedirs(IMGS_DIR, exist_ok=True)
makedirs(SAT_IMGS_DIR, exist_ok=True)

In [ ]:
with open("./satellites/data/fortaleza_renda_hex.geojson", "r") as ifp:
  hex_geo = sorted(json.load(ifp)["features"], key=lambda x: int(x["properties"]["id"]))

In [ ]:
img_data = []

for cnt,hx in enumerate(hex_geo):
  if cnt % 20 == 0:
    print(cnt, "/", len(hex_geo))

  hx_id = int(hx["properties"]["id"])
  img_id = f"{pad(hx_id, 5)}"

  coords = np.array(hx["geometry"]["coordinates"][0])
  coords_center = coords.mean(axis=0)
  coords_start = coords.min(axis=0) # bottom-left
  coords_stop = coords.max(axis=0) # top-right

  filepath = f"{SAT_IMGS_DIR}/{run_name}_{img_id}.jpg"

  if path.isfile(filepath):
    continue

  MAPS_PARAMS["visible"] = f"{coords_start[1]},{coords_start[0]}|{coords_stop[1]},{coords_stop[0]}"

  response = requests.get(MAPS_URL, headers=MAPS_HEADER, params=MAPS_PARAMS, stream=True)
  PImage.open(response.raw).save(filepath)
  sleep(0.25)

### Combine Images

In [ ]:
with open("./satellites/data/fortaleza_renda_hex.geojson", "r") as ifp:
  hex_geo = sorted(json.load(ifp)["features"], key=lambda x: int(x["properties"]["id"]))

coords = np.array([hx["geometry"]["coordinates"][0] for hx in hex_geo])

In [ ]:
# figure out how many images to use to tile

coords_min = coords.min(axis=0).min(axis=0)
coords_max = coords.max(axis=0).max(axis=0)
coords_diff = coords_max - coords_min

coords_avgs = coords.mean(axis=1)
coords_mins = coords.min(axis=1)
coords_maxs = coords.max(axis=1)

avg_diffs = np.concat((coords_maxs - coords_avgs, coords_avgs - coords_mins), axis=0).mean(axis=0)

In [ ]:
img_w = 6400
nimgs = (coords_diff / avg_diffs).astype(int)
hex_dim = int(img_w / max(nimgs) * 2.75)
img_h = int(img_w * nimgs[1] / nimgs[0] * 0.93)

img_prefix = f"sat_hex"
img_dir = f"satellites/imgs/{img_prefix}"

fimg = PImage.new("RGB", (img_w, img_h))

for cnt,hx in enumerate(hex_geo):
  hx_id = int(hx["properties"]["id"])
  img_id = f"{pad(hx_id, 5)}"
  img = PImage.open(f"{img_dir}/{img_prefix}_{img_id}.jpg").resize((hex_dim, hex_dim))

  coords = np.array(hx["geometry"]["coordinates"][0])
  coords_center = coords.mean(axis=0)
  coords_start = coords.min(axis=0) # bottom-left
  coords_stop = coords.max(axis=0) # top-right

  x = (coords_start[0] - coords_min[0]) / (coords_max[0] - coords_min[0]) * fimg.width
  y = fimg.height - (coords_stop[1] - coords_min[1]) / (coords_max[1] - coords_min[1]) * fimg.height

  fimg.paste(img, (int(x), int(y)))
  img.close()

fimg.save(f"satellites/imgs/{img_prefix}.jpg")
display(fimg)

## CV Analysis

In [ ]:
!git clone --depth 1 https://github.com/direito-a-sombra/bus-view-satellites.git satellites

In [ ]:
import json

from os import makedirs, path
from PIL import Image as PImage

from torch import cuda, no_grad
from torchvision import transforms as T
from transformers import MaskFormerForInstanceSegmentation, MaskFormerImageProcessor

from env import HFTOKEN

def pad(n, l=3):
  return (l*"0" + str(n))[-l:] if n < 10**l else str(n)

VERSION = f"sat_hex"
DATA_DIR = f"satellites/data"
IMGS_DIR = f"satellites/imgs/{VERSION}"

MODEL_ID = "thiagohersan/maskformer-satellite-trees"

In [ ]:
ade_mean = [ 0.485, 0.456, 0.406 ]
ade_std = [ 0.229, 0.224, 0.225 ]

ade_transform = T.Compose([
  T.ToTensor(),
  T.Normalize(mean=ade_mean, std=ade_std)
])

preprocessor = MaskFormerImageProcessor(
  do_resize=False,
  do_normalize=False,
  do_rescale=False,
  ignore_index=255,
  reduce_labels=False
)

device = "cuda" if cuda.is_available() else "cpu"
model = MaskFormerForInstanceSegmentation.from_pretrained(MODEL_ID, token=HFTOKEN).to(device)
included_labels = [l for l in model.config.id2label.values() if l not in ["other", "unknown"]]

In [ ]:
with open(f"{DATA_DIR}/fortaleza_renda_hex.geojson", "r") as ifp:
  hex_geo = json.load(ifp)

hex_geo["features"] = sorted(hex_geo["features"], key=lambda x: int(x["properties"]["id"]))

for hx in hex_geo["features"]:
  for l in included_labels:
    hx["properties"][f"{l}_pct"] = 0.0

In [ ]:
for cnt,hx in enumerate(hex_geo["features"]):
  hx_id = int(hx["properties"]["id"])
  img_id = f"{pad(hx_id, 5)}"

  img = PImage.open(f"{IMGS_DIR}/{VERSION}_{img_id}.jpg")
  iw,ih = img.size
  inputs = preprocessor(images=ade_transform(img), return_tensors="pt").to(device)

  with no_grad():
    outputs = model(**inputs)
    results = preprocessor.post_process_semantic_segmentation(outputs=outputs, target_sizes=[(ih, iw)])[0]

  label_ids, label_cnts = results.cpu().unique(return_counts=True)
  id2count = { int(lid): int(cnt) for lid, cnt in zip(label_ids, label_cnts) }

  for lid in label_ids.tolist():
    label = model.config.id2label[lid]
    if label in included_labels:
      hx["properties"][f"{label}_pct"] = round(id2count[lid] / (iw * ih), 4)

In [ ]:
with open(f"{DATA_DIR}/{VERSION}.geojson", "w") as ofp:
  json.dump(hex_geo, ofp, ensure_ascii=False)